# Campos anuláveis (nullable)

Um campo anulável é um pouco diferente de um campo opcional.

Campo opcional significa apenas que os dados desserializados não precisam conter a **chave** daquele campo; nesse caso, usa-se um
default predefinido.

Já a anulabilidade não tem nada a ver com o campo ser opcional ou não: ela só indica se o campo pode receber `None` (ou `null` em JSON).

Vejamos:

Como o Pydantic valida tipos, se definimos um campo como, digamos, `int`, tentar atribuir `None` falha, porque `None` é um tipo
distinto em Python e não pode ser convertido para int.

In [1]:
from pydantic import BaseModel, ValidationError

class Model(BaseModel):
    field: int

In [2]:
try:
    Model(field=None)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
field
  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/int_type


A exceção diz que o dado não é um inteiro válido — bem diferente desta:

In [3]:
try:
    Model()
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
field
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


em que a exceção é sobre campo obrigatório ausente.

Para dizer ao Pydantic que um campo é anulável (aceita o tipo real **ou** `None`), precisamos especificar isso no *type hint*.

In [4]:
class Model(BaseModel):
    field: int | None

Esse *type hint* diz ao Pydantic que um inteiro (ou algo conversível para inteiro) ou `None` são valores aceitáveis.

In [5]:
print(f"Model(field=None): {Model(field=None)!r}")

Model(field=None): Model(field=None)


In [6]:
print(f'Model(field="1"): {Model(field="1")!r}')

Model(field="1"): Model(field=1)


Agora temos um campo anulável.

Mas o campo **não** é opcional!

In [7]:
try:
    Model()
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
field
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


Campos anuláveis e opcionais costumam andar juntos, porque é comum usar `None` como default para indicar que nenhum valor foi fornecido.

Nesse caso, queremos o campo **ao mesmo tempo** anulável e opcional.

Fazemos assim:

In [8]:
class Model(BaseModel):
    field: int | None = None

Agora o campo é opcional (porque tem default) e anulável (por causa do *type hint* `int | None`).

Escolhemos `None` como default, mas poderia ser qualquer inteiro.

A notação `int | None` só existe a partir do Python 3.10.

Antes disso, havia (e ainda há) duas outras formas de escrever esse *type hint*.

A primeira é usar `Union` do módulo `typing`:

In [9]:
from typing import Union

class Model(BaseModel):
    field: Union[int, None]

É totalmente equivalente a `int | None`.

In [10]:
print(f"Model(field=None): {Model(field=None)!r}")

Model(field=None): Model(field=None)


A outra forma é usar `Optional`, também de `typing`:

In [11]:
from typing import Optional

class Model(BaseModel):
    field: Optional[int]

Eu evito `Optional`, mesmo em versões anteriores ao Python 3.10.

O motivo: em modelos Pydantic, "opcional" significa outra coisa. Aqui `Optional` indica que o campo é **anulável**, não que é opcional.

Podemos verificar:

In [12]:
try:
    Model()
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
field
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


Para tornar o campo opcional, precisamos de um default, assim:

In [13]:
class Model(BaseModel):
    field: Optional[int] = None

In [14]:
print(f"Model(): {Model()!r}")

Model(): Model(field=None)


As três formas de especificar anulabilidade são totalmente equivalentes:

In [15]:
class Model(BaseModel):
    field_1: int | None
    field_2: Union[int, None]
    field_3: Optional[int]

E, olhando as definições dos campos:

In [16]:
print(f"Model.model_fields: {Model.model_fields!r}")

Model.model_fields: {'field_1': FieldInfo(annotation=Union[int, NoneType], required=True), 'field_2': FieldInfo(annotation=Union[int, NoneType], required=True), 'field_3': FieldInfo(annotation=Union[int, NoneType], required=True)}


Os três campos aparecem como `Union[int, NoneType]`.

Tanto `int | None` quanto `Optional[int]` são apenas sintaxe de **conveniência** para `Union[int, None]`.

Um erro muito comum de iniciantes é dar default `None` sem indicar que o campo é anulável.

Ou seja, é comum ver código assim:

In [17]:
class Model(BaseModel):
    field: int = None

Como o Pydantic (por padrão) não valida defaults, o modelo parece funcionar, mas pode causar problemas depois:

In [18]:
m = Model()
print(f"m: {m!r}")

m: Model(field=None)


Mas o campo não é anulável de verdade, então isto não funciona:

In [19]:
try:
    Model(field=None)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
field
  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/int_type


O jeito correto de definir esse modelo seria:

In [20]:
class Model(BaseModel):
    field: int | None = None

Vamos passar mais um tempo nas combinações de opcional e anulável — próxima aula!